# Exploratory Data Analysis

Situation 2 ("Rating Wars"): before building a weighted rating, look at how the plain, unweighted rating behaves and how the helpfulness votes relate to it.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from data_cleaning import DATA_PATH, load_and_clean

PLOTS_DIR = Path("plots")
PLOTS_DIR.mkdir(exist_ok=True)

## Raw data check

Look at the CSV as it comes, before any cleaning, to find what needs fixing.

In [ ]:
raw = pd.read_csv(DATA_PATH)
print(raw.shape)
raw.dtypes

In [ ]:
pd.DataFrame({"missing": raw.isna().sum()}).assign(duplicate_rows=raw.duplicated().sum())

User's Rating out of 10 was read as **text**, not a number. Its values show why:

In [ ]:
raw["User's Rating out of 10"].str.strip().value_counts()

In [ ]:
raw[["Usefulness Vote", "Total Votes"]].describe().round(1)

In [ ]:
print("Reviews with 0 total votes:", (raw["Total Votes"] == 0).sum())
print("Useful > total (impossible):", (raw["Usefulness Vote"] > raw["Total Votes"]).sum())
print("Unique users:", raw["User"].nunique(), "for", len(raw), "reviews")

**Findings**

- 2,961 reviews, 7 columns, no missing values and no duplicate rows.
- **200 ratings are not numbers** – the cell holds scraped page text (*"Was this review helpful? Sign in to vote."*). These rows have no usable rating.
- `Date of Review` is text (e.g. *"13 March 2003"*) and has to be parsed as a date.
- Vote counts are valid (useful ≤ total everywhere) but very skewed: most reviews have 0–2 votes, the top one has 1,171.
- Almost every user wrote one review, so quality has to be judged per review, not per user.

**Cleaning (`data_cleaning.py`)**: rename columns, parse dates and numbers, drop the 200 rows without a numeric rating, strip text and add `review_length` (word count).

## Clean data

In [ ]:
df = load_and_clean()
df.head()

## Plain rating

In [ ]:
pd.Series({
    "reviews": len(df),
    "plain_average": df["rating"].mean(),
    "median": df["rating"].median(),
    "share_10": (df["rating"] == 10).mean(),
    "share_1": (df["rating"] == 1).mean(),
}).round(3)

In [ ]:
counts = df["rating"].value_counts().reindex(range(1, 11), fill_value=0)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(counts.index, counts.values, color="#4C72B0")
for x, y in zip(counts.index, counts.values):
    ax.text(x, y, str(y), ha="center", va="bottom", fontsize=9)
ax.axvline(df["rating"].mean(), color="#C44E52", linestyle="--",
           label=f"plain average = {df['rating'].mean():.2f}")
ax.set_xticks(range(1, 11))
ax.set_xlabel("User's rating (out of 10)")
ax.set_ylabel("Number of reviews")
ax.set_title("Rating distribution")
ax.legend()
fig.tight_layout()
# fig.savefig(PLOTS_DIR / "rating_distribution.png", dpi=120)
plt.show()

The plain average is **8.85** with a median of 10. Ratings are extremely polarized: almost 59% of reviews are 10s, while 1–7 together are a thin tail with a small bump at 1.

## Votes by rating

`Total Votes` = readers who answered *"Was this review helpful?"*; `Usefulness Vote` = those who answered *yes*. `helpful_share` = useful votes / total votes for all reviews with that rating.

In [ ]:
g = df.groupby("rating")
pd.DataFrame({
    "reviews": g.size(),
    "avg_total_votes": g["total_votes"].mean(),
    "helpful_share": g["useful_votes"].sum() / g["total_votes"].sum(),
}).round(2)

- **1-star reviews attract the most votes** (≈68 on average) but only **35%** of them are *helpful* – readers actively push back on them.
- **10-star reviews** are judged helpful **71%** of the time.
- So the raw vote count alone is not a quality signal (controversial reviews get many votes); the helpful *share* has to be part of the weight.

## Ratings over time

In [ ]:
yearly = df.groupby(df["date"].dt.year).agg(
    reviews=("rating", "size"),
    average=("rating", "mean"),
    ones=("rating", lambda r: (r == 1).sum()),
)

fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.bar(yearly.index, yearly["reviews"], color="#BBBBBB", label="all reviews")
ax1.bar(yearly.index, yearly["ones"], color="#C44E52", label="1-star reviews")
ax1.set_xlabel("Year")
ax1.set_ylabel("Reviews per year")

ax2 = ax1.twinx()
ax2.plot(yearly.index, yearly["average"], color="#4C72B0", marker="o", label="average rating")
ax2.set_ylim(1, 10)
ax2.set_ylabel("Average rating")

handles = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
ax1.legend(handles, [h.get_label() for h in handles], loc="upper left")
ax1.set_title("Reviews and average rating per year")
fig.tight_layout()
# fig.savefig(PLOTS_DIR / "ratings_over_time.png", dpi=120)
plt.show()

- The yearly average climbs from ≈7.3 (1998) to ≈9 (2011 onward).
- Review volume jumps in **2018–2021** (peak ≈440 in 2020), almost entirely high ratings, while 1-star reviews stay at a handful per year.
- In this dataset the "rating war" shows up as a **wave of 10s** rather than a hater attack. Most of these recent reviews have 0–2 votes, so a helpfulness-based weight should reduce their influence. A 1-star attack is simulated later in step 5.